In [10]:
import os
import random
import time

import gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Imports all our hyperparameters from the other file
from hyperparams import Hyperparameters as params

print("Replay buffer size:", params.buffer_size)
print("Using buffer size from params:", params.buffer_size)

Replay buffer size: 100000
Using buffer size from params: 100000


In [11]:
# stable_baselines3 have wrappers that simplifies 
# the preprocessing a lot, read more about them here:
# https://stable-baselines3.readthedocs.io/en/master/common/atari_wrappers.html
from stable_baselines3.common.atari_wrappers import (
    ClipRewardEnv,
    EpisodicLifeEnv,
    FireResetEnv,
    MaxAndSkipEnv,
    NoopResetEnv,
)


In [12]:
from stable_baselines3.common.buffers import ReplayBuffer
# Creates our gym environment and with all our wrappers.
def make_env(env_id, seed, idx, capture_video, run_name):
    def thunk():
        env = gym.make(env_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        if capture_video:
            if idx == 0:
                env = gym.wrappers.RecordVideo(env, f"videos/{run_name}")
        env = NoopResetEnv(env, noop_max=30) # no op between 0 and 30 to give the agent some randomness for wen it starts acting
        env = MaxAndSkipEnv(env, skip=4) # skip 4 timesteps befor making anew decition
        env = EpisodicLifeEnv(env)
        if "FIRE" in env.unwrapped.get_action_meanings():
            env = FireResetEnv(env)
        env = ClipRewardEnv(env)
        env = gym.wrappers.ResizeObservation(env, (84, 84)) # rescale the image
        env = gym.wrappers.GrayScaleObservation(env) # gray scale
        env = gym.wrappers.FrameStack(env, 4) # look at the 4 previous images to make a decition
        env.seed(seed)
        env.action_space.seed(seed)
        env.observation_space.seed(seed)
        return env

    return thunk

In [13]:
class QNetwork(nn.Module):
    def __init__(self, env):
        super().__init__()
        # Get number of actions
        n_actions = env.single_action_space.n
        # Get input shape from observation space
        obs_shape = env.single_observation_space.shape  # e.g., (4, 84, 84)
        in_channels = obs_shape[0]

        self.network = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(7 * 7 * 64, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def forward(self, x):
        return self.network(x / 255.0)


In [14]:
def linear_schedule(start_e: float, end_e: float, duration: int, t: int):
    slope = (end_e - start_e) / duration
    return max(slope * t + start_e, end_e)

In [15]:
import os
os.environ["WANDB_DISABLE_GYM"] = "true"
import wandb

wandb.init(
    project="your-project-name",   # ← change this to your W&B project
    name='QDN_ELiasFalk',
    config={
        "env_id": params.env_id,
        "learning_rate": params.learning_rate,
        "gamma": params.gamma,
        "batch_size": params.batch_size,
        "buffer_size": params.buffer_size,
        "exploration_fraction": params.exploration_fraction,
        "target_network_frequency": params.target_network_frequency,
        "tau": params.tau,
        "seed": params.seed,
    }
)

charts/episodic_return,▁▄▁▃▁▃▄▆▂▃▄▆▂▅▂▁▆▁▁▁█▁▅▃▄▁▂▁▁▁▂▁▁▄▃▃▃▁▃▁
loss/td_loss,▁▁▁▇▁▁█▁▁▁▁▁▁▁▁▁▁█▁█▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▂▁▁██
charts/episodic_return,4
loss/td_loss,0.06296


C:\Users\elias\Documents\Anaconda\envs\ex6\lib\site-packages\wandb\analytics\sentry.py:259: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}  # noqa


In [16]:
if __name__ == "__main__":
    run_name = f"{params.env_id}__{params.exp_name}__{params.seed}__{int(time.time())}"

    random.seed(params.seed)
    np.random.seed(params.seed)
    torch.manual_seed(params.seed)
    torch.backends.cudnn.deterministic = params.torch_deterministic

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)
    # env setup
    envs = gym.vector.SyncVectorEnv([make_env(params.env_id, params.seed, 0, params.capture_video, run_name)])
    assert isinstance(envs.single_action_space, gym.spaces.Discrete), "only discrete action space is supported"

    #setting up the network and hyperparameters
    q_network = QNetwork(envs).to(device)
    optimizer = optim.Adam(q_network.parameters(), lr=params.learning_rate)
    target_network = QNetwork(envs).to(device) # setting upp the network we are targeting
    target_network.load_state_dict(q_network.state_dict())
    
    # We’ll be using experience replay memory for training our DQN. 
    # It stores the transitions that the agent observes, allowing us to reuse this data later. 
    # By sampling from it randomly, the transitions that build up a batch are decorrelated. 
    # It has been shown that this greatly stabilizes and improves the DQN training procedure.
    print("Replay buffer size:", params.buffer_size)
    rb = ReplayBuffer(
        params.buffer_size,
        envs.single_observation_space,
        envs.single_action_space,
        device,
        optimize_memory_usage=False,
        handle_timeout_termination=True,
    )

    obs = envs.reset() 
    for global_step in range(params.total_timesteps):
        # Here we get epsilon for our epislon greedy.
        epsilon = linear_schedule(params.start_e, params.end_e, params.exploration_fraction * params.total_timesteps, global_step)
        if random.random() < epsilon: # This takes a random action if true
            actions = np.array([envs.single_action_space.sample()])
        else: #Here the Q-network decides what to do
            obs_tensor = torch.tensor(obs, dtype=torch.float32).to(device)
            q_values = q_network(obs_tensor)
            actions = torch.argmax(q_values, dim=1).cpu().numpy()


        # Take a step in the environment
        next_obs, rewards, dones, infos = envs.step(actions)

        # Here we print our reward.
        for info in infos:
            if "episode" in info.keys():
                episodic_return = info['episode']['r']
                episode_length = info['episode']['l']
                wandb.log({
                    "charts/episodic_return": episodic_return,
                    "charts/episode_length": episode_length,
                    "exploration/epsilon": epsilon,
                }, step=global_step)
                break


        # Save data to replay buffer
        real_next_obs = next_obs.copy()
        for idx, d in enumerate(dones):
            if d:
                real_next_obs[idx] = infos[idx]["terminal_observation"]

        # Here we store the transitions in D
        rb.add(obs, real_next_obs, actions, rewards, dones, infos)

        obs = next_obs
        # Training 
        if global_step > params.learning_starts:
            data = rb.sample(params.batch_size) # sample from the replay_buffer 32
            

            # Compute Q-values for current states
            q_values = q_network(data.observations)               # Shape: [batch_size, num_actions]
            action_q_values = q_values.gather(1, data.actions.long()).squeeze(1)  # Shape: [batch_size]

            # Compute TD target using the target network (no gradients)
            with torch.no_grad():
                next_q_values = target_network(data.next_observations)           # Shape: [batch_size, num_actions]
                max_next_q_values = next_q_values.max(1)[0]                      # Shape: [batch_size]
                td_target = data.rewards + params.gamma * (1 - data.dones) * max_next_q_values  # Shape: [batch_size] It makes shure that if the episode ended it does not consider future q values

            # Compute the loss
            loss = F.mse_loss(action_q_values, td_target)
            wandb.log({"loss/td_loss": loss.item()}, step=global_step)


            # Perform gradient descent
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # update target network
            if global_step % params.target_network_frequency == 0:
                for target_network_param, q_network_param in zip(target_network.parameters(), q_network.parameters()):
                    target_network_param.data.copy_(params.tau * q_network_param.data + (1.0 - params.tau) * target_network_param.data)
                     #This just makes our Q network the target network every 1000 iterations

            


    if params.save_model:
        model_path = f"runs/{run_name}/{params.exp_name}_model"
        torch.save(q_network.state_dict(), model_path)
        print(f"model saved to {model_path}")

    envs.close()


cuda


C:\Users\elias\Documents\Anaconda\envs\ex6\lib\site-packages\gym\utils\seeding.py:138: DeprecationWarning: WARN: Function `hash_seed(seed, max_bytes)` is marked as deprecated and will be removed in the future. 
  deprecation(
C:\Users\elias\Documents\Anaconda\envs\ex6\lib\site-packages\gym\utils\seeding.py:175: DeprecationWarning: WARN: Function `_bigint_from_bytes(bytes)` is marked as deprecated and will be removed in the future. 
  deprecation(


Replay buffer size: 100000


C:\Users\elias\Documents\Anaconda\envs\ex6\lib\site-packages\gym\wrappers\monitoring\video_recorder.py:43: DeprecationWarning: WARN: `env.metadata["render.modes"] is marked as deprecated and will be replaced with `env.metadata["render_modes"]` see https://github.com/openai/gym/pull/2654 for more details
  logger.deprecation(
C:\Users\elias\Documents\Anaconda\envs\ex6\lib\site-packages\gym\wrappers\monitoring\video_recorder.py:341: DeprecationWarning: Use shutil.which instead of find_executable
  if distutils.spawn.find_executable("avconv") is not None:
C:\Users\elias\Documents\Anaconda\envs\ex6\lib\site-packages\gym\wrappers\monitoring\video_recorder.py:421: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if distutils.version.LooseVersion(
C:\Users\elias\Documents\Anaconda\envs\ex6\lib\site-packages\gym\utils\seeding.py:47: DeprecationWarning: WARN: Function `rng.randint(low, [high, size, dtype])` is marked as deprecated and will be remove

global_step=109, episodic_return=0.0
global_step=250, episodic_return=1.0
global_step=365, episodic_return=0.0
global_step=573, episodic_return=2.0
global_step=856, episodic_return=4.0
global_step=1042, episodic_return=2.0
global_step=1299, episodic_return=3.0
global_step=1460, episodic_return=1.0
global_step=1598, episodic_return=1.0
global_step=1711, episodic_return=0.0
global_step=1824, episodic_return=0.0
global_step=2036, episodic_return=2.0
global_step=2341, episodic_return=4.0
global_step=2550, episodic_return=2.0
global_step=2711, episodic_return=1.0
global_step=2949, episodic_return=3.0
global_step=3108, episodic_return=1.0
global_step=3344, episodic_return=3.0
global_step=3532, episodic_return=2.0
global_step=3754, episodic_return=3.0
global_step=3867, episodic_return=0.0
global_step=3982, episodic_return=0.0
global_step=4191, episodic_return=2.0
global_step=4456, episodic_return=4.0
global_step=4569, episodic_return=0.0
global_step=4682, episodic_return=0.0
global_step=4793,

C:\Users\elias\AppData\Local\Temp\ipykernel_62752\518337888.py:86: UserWarning: Using a target size (torch.Size([32, 32])) that is different to the input size (torch.Size([32])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(action_q_values, td_target)


global_step=80005, episodic_return=3.0
global_step=80241, episodic_return=3.0
global_step=80400, episodic_return=1.0
global_step=80562, episodic_return=1.0
global_step=80704, episodic_return=1.0
global_step=80817, episodic_return=0.0
global_step=80932, episodic_return=0.0
global_step=81045, episodic_return=0.0
global_step=81231, episodic_return=2.0
global_step=81441, episodic_return=2.0
global_step=81556, episodic_return=0.0
global_step=81747, episodic_return=2.0
global_step=81935, episodic_return=2.0
global_step=82201, episodic_return=4.0
global_step=82469, episodic_return=4.0
global_step=82677, episodic_return=2.0
global_step=82864, episodic_return=2.0
global_step=82977, episodic_return=0.0
global_step=83310, episodic_return=5.0
global_step=83495, episodic_return=2.0
global_step=83714, episodic_return=3.0
global_step=83872, episodic_return=1.0
global_step=83983, episodic_return=0.0
global_step=84096, episodic_return=0.0
global_step=84410, episodic_return=5.0
global_step=84523, episod

KeyboardInterrupt: 

In [ ]:
print("q_values.shape:", q_values.shape)
print("data.actions.shape:", data.actions.shape)
print("unsqueezed actions shape:", data.actions.long().unsqueeze(-1).shape)
print("unsqueezed actions shape:", q_values.gather(1, data.actions.long()).squeeze(1).shape)

In [ ]:
q_values = q_network(data.observations)  # [32, 4]
action_q_values = q_values.gather(1, data.actions.long()).squeeze(1)  # [32]

In [ ]:
print(f"Q-Network is on: {next(q_network.parameters()).device}")

In [ ]:
print("Is CUDA available?", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("Current CUDA device:", torch.cuda.current_device())
print("CUDA device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

In [ ]:
if torch.cuda.is_available():
    print("CUDA is available")
    print("Number of GPUs:", torch.cuda.device_count())
    print("Current GPU device:", torch.cuda.current_device())
    print("GPU device name:", torch.cuda.get_device_name(torch.cuda.current_device()))
else:
    print("CUDA is not available")

In [ ]:
import os
import random
import time

import gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim